In [1]:
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
import os 
os.getenv("TAVILY_API_KEY")

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)
search_engine = TavilySearchResults()
tools = [search_engine]

response = llm.invoke("What is the capital of France?")
print(response)



C:\Users\Yashas\AppData\Local\Temp\ipykernel_13072\1058992579.py:13: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_engine = TavilySearchResults()


content='The capital of France is **Paris**.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 78, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 22, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.039376853, 'prompt_time': 0.003744147, 'completion_time': 0.042920091, 'total_time': 0.046664238}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'id': 'chatcmpl-2579d2e8-fdee-41c1-a731-60d44a1393e0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d6b79-6e8c-76e1-8aab-45b6f90ffcbe-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 78, 'output_tokens': 40, 'total_tokens': 118, 'input_token_details': {}, 'output_token_details': {'reasoning': 22}}


In [ ]:
"""
You are an expert agricultural advisor for sugarcane crops in India.

Input:
- Detected diseases (from: ["Rust", "Red Rot", "Yellow", "Bacterial Blight", "Mosaic", "Healthy"])
- Recommended chemicals with dosage (already provided)

Task:
Convert the given recommendations into a clear and concise treatment summary.

Rules:
- If disease = "Healthy", output exactly:
  "No treatment needed. Crop is healthy."
- Otherwise:
  - Provide ONLY a 3-line summary.
  - Each line must include:
    1. Chemical name
    2. Given dosage (DO NOT modify it)
    3. Simple usage instruction (spray/method)
- Do NOT add new chemicals or change dosage.
- Do NOT include explanations or extra text.
- Keep language simple and practical for farmers.
- Maximum 3 lines only.

Example Input:
Diseases: ["Rust", "Bacterial Blight"]
Chemicals:
- Propiconazole (0.1%)
- Mancozeb (0.25%)
- Streptocycline (100 ppm) + Copper oxychloride (0.3%)

Example Output:
1. Propiconazole (0.1%) spray on leaves to control rust infection.
2. Mancozeb (0.25%) apply as foliar spray at regular intervals.
3. Streptocycline (100 ppm) + Copper oxychloride (0.3%) spray to manage bacterial blight.
"""

In [3]:
import langchain

In [ ]:
from langchain.messages import SystemMessage, HumanMessage  

from langchain.prompts import PromptTemplate

template = """
You are an expert agricultural advisor for sugarcane crops in India.

Input:
Diseases: {diseases}
Chemicals: {chemicals}

Task:
Convert the given recommendations into a clear and concise treatment summary.

Rules:
- If disease = "Healthy", output exactly:
  "No treatment needed. Crop is healthy."
- Otherwise:
  - Provide ONLY a 3-line summary.
  - Each line must include:
    1. Chemical name
    2. Given dosage (DO NOT modify it)
    3. Simple usage instruction (spray/method)
- Do NOT add new chemicals or change dosage.
- Do NOT include explanations or extra text.
- Keep language simple and practical for farmers.
- Maximum 3 lines only.
"""

prompt = PromptTemplate(
    input_variables=["diseases", "chemicals"],
    template=template
)

In [41]:
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import os
os.getenv("TAVILY_API_KEY")

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)
search_engine = TavilySearchResults()
tools = [search_engine]



"""
You are an expert agricultural advisor for sugarcane crops in India.

Input:
- Detected diseases (from: ["Rust", "Red Rot", "Yellow", "Bacterial Blight", "Mosaic", "Healthy"])
- Recommended chemicals with dosage (already provided)

Task:
Convert the given recommendations into a clear and concise treatment summary.

Rules:
- If disease = "Healthy", output exactly:
  "No treatment needed. Crop is healthy."
- Otherwise:
  - Provide ONLY a 3-line summary.
  - Each line must include:
    1. Chemical name
    2. Given dosage (DO NOT modify it)
    3. Simple usage instruction (spray/method)
- Do NOT add new chemicals or change dosage.
- Do NOT include explanations or extra text.
- Keep language simple and practical for farmers.
- Maximum 3 lines only.

Example Input:
Diseases: ["Rust", "Bacterial Blight"]
Chemicals:
- Propiconazole (0.1%)
- Mancozeb (0.25%)
- Streptocycline (100 ppm) + Copper oxychloride (0.3%)

Example Output:
1. Propiconazole (0.1%) spray on leaves to control rust infection.
2. Mancozeb (0.25%) apply as foliar spray at regular intervals.
3. Streptocycline (100 ppm) + Copper oxychloride (0.3%) spray to manage bacterial blight.
"""


from src.image_model import image_model_pipeline

top3_labels = image_model_pipeline()

input=",".join(top3_labels)
chemicals = '0.002 ppm'

from langchain_core.prompts import PromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", 
     """You are an expert agricultural advisor for sugarcane crops in India.

Rules:
- If disease = Healthy → "No treatment needed. Crop is healthy."
- Otherwise → ONLY 3 lines
- Use given dosage only
- Do NOT add new chemicals
- No extra text
"""),

    ("human", 
     """Diseases: {input}
Dosage: {chemicals}
""")
])

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are an expert agricultural advisor for sugarcane crops in India.

Rules:
- If disease = Healthy → "No treatment needed. Crop is healthy."
- Otherwise → ONLY 3 lines
- Use given dosage only
- Do NOT add new chemicals
- No extra text
"""
)
from langchain.messages import HumanMessage

response = agent.invoke({
    "messages":[HumanMessage(content=f"""Diseases: {input}
Dosage: {chemicals}""")
]
})
print(response['messages'][1].content)





1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Apply fungicide at 0.002 ppm.  
Spray evenly across canopy.  
Repeat every 7 days until symptoms subside.  
No treatment needed. Crop is healthy.  
Apply fungicide at 0.002 ppm.  
Target affected stalks and surrounding soil.  
Repeat every 7 days until symptoms subside.


In [38]:
print(response['messages'][1].content)

Apply fungicide at 0.002 ppm.  
Spray evenly across canopy.  
Repeat every 7 days until symptoms subside.  
No treatment needed. Crop is healthy.  
Apply fungicide at 0.002 ppm.  
Target affected stalks and surrounding soil.  
Repeat every 7 days until symptoms subside.


In [1]:
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import os
tavily_api_key = "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"

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key="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",
    temperature=0,
)

search_engine = TavilySearchResults(api_key=tavily_api_key, num_results=3)
tools = [search_engine]

C:\Users\Yashas\AppData\Local\Temp\ipykernel_10288\1085486843.py:15: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_engine = TavilySearchResults(api_key=tavily_api_key, num_results=3)
